# PortfolioVectorEngine dummy walkthrough

This notebook uses tiny artificial price and signal data to show exactly what happens inside `PortfolioVectorEngine.run()`.

The key formulas in the current engine are:

```python
self.prices, self.signals = prices.align(signals, join="inner")
market_returns = self.prices.pct_change(fill_method=None).fillna(0)
actual_weights = self.signals.shift(self.delay).fillna(0)
portfolio_gross_returns = (actual_weights.shift(1) * market_returns).sum(axis=1)
gross_equity = initial_capital * (1 + portfolio_gross_returns).cumprod()
target_allocation_usd = actual_weights.multiply(gross_equity, axis=0)
target_shares = target_allocation_usd / self.prices
share_diff = target_shares.diff().fillna(0)
costs = fee_model.calculate_vector_matrix(asset_class, share_diff, self.prices)
net_equity = gross_equity - cumulative_costs
```

The most important part for execution timing is that there are two separate shifts: `signals.shift(delay)` and then `actual_weights.shift(1)` when multiplying returns.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.vector_backtester_Mateo import PortfolioVectorEngine

pd.set_option("display.float_format", "{:.6f}".format)
pd.set_option("display.max_columns", 50)

## Helper functions

The next cell recreates the engine calculations step by step. It is intentionally verbose, because the goal is to inspect timing and arithmetic.

In [ ]:
class ZeroFeeModel:
    def calculate_vector_matrix(self, asset_class, share_diff, prices):
        return share_diff.abs() * 0.0


def run_engine(prices, signals, delay, initial_capital=100_000, asset_class="STK", zero_fees=True):
    engine = PortfolioVectorEngine(
        prices=prices,
        signals=signals,
        asset_class=asset_class,
        initial_capital=initial_capital,
        execution_delay=delay,
    )
    if zero_fees:
        engine.fee_model = ZeroFeeModel()
    stats = engine.run()
    return engine, stats


def explain_engine(prices, signals, delay, initial_capital=100_000, asset_class="STK", zero_fees=True):
    engine, stats = run_engine(
        prices=prices,
        signals=signals,
        delay=delay,
        initial_capital=initial_capital,
        asset_class=asset_class,
        zero_fees=zero_fees,
    )

    aligned_prices = engine.prices
    aligned_signals = engine.signals
    market_returns = aligned_prices.pct_change(fill_method=None).fillna(0)
    actual_weights = aligned_signals.shift(delay).fillna(0)
    weight_used_for_return = actual_weights.shift(1)
    gross_return_contrib = weight_used_for_return * market_returns
    target_allocation_usd = actual_weights.multiply(stats["gross_equity"], axis=0)  # VOY POR AQUI ==================================================================
    target_shares = target_allocation_usd / aligned_prices
    share_diff = target_shares.diff().fillna(0)

    if len(aligned_prices.columns) == 1:
        symbol = aligned_prices.columns[0]
        detail = pd.DataFrame({
            "price_open": aligned_prices[symbol],
            "raw_signal_weight": aligned_signals[symbol],
            "market_return_from_prev_open": market_returns[symbol],
            "actual_weight_after_delay": actual_weights[symbol],
            "weight_used_for_this_return": weight_used_for_return[symbol],
            "gross_return_contribution": gross_return_contrib[symbol],
            "portfolio_gross_returns": stats["gross_returns"],
            "gross_equity": stats["gross_equity"],
            "target_allocation_usd": target_allocation_usd[symbol],
            "target_shares": target_shares[symbol],
            "share_diff": share_diff[symbol],
            "total_costs_usd": stats["total_costs_usd"],
            "net_equity": stats["net_equity"],
        })
    else:
        detail = {
            "aligned_prices": aligned_prices,
            "aligned_signals": aligned_signals,
            "market_returns": market_returns,
            "actual_weights_after_delay": actual_weights,
            "weights_used_for_returns": weight_used_for_return,
            "gross_return_contributions": gross_return_contrib,
            "target_shares": target_shares,
            "share_diff": share_diff,
            "portfolio_stats": stats,
        }

    return engine, stats, detail

In [6]:
def play_sound(name="Glass"):
    import subprocess
    subprocess.run(["afplay", f"/System/Library/Sounds/{name}.aiff"])

play_sound()
play_sound("Ping")

## Example 1: one asset, zero fees, `execution_delay=0`

These prices are indexed by the time of the open. Therefore the return shown on `2024-01-02` is the return from the `2024-01-01` open to the `2024-01-02` open.

With `execution_delay=0`, a signal stamped on `2024-01-01` is used for the return row on `2024-01-02`.

In [3]:
dates = pd.to_datetime([
    "2024-01-01",
    "2024-01-02",
    "2024-01-03",
    "2024-01-04",
    "2024-01-05",
])

prices_1 = pd.DataFrame({"ABC": [100, 110, 99, 120, 60]}, index=dates)
signals_1 = pd.DataFrame({"ABC": [1.0, 1.0, -1.0, 0.0, 1.0]}, index=dates)

engine0, stats0, detail0 = explain_engine(prices_1, signals_1, delay=0, zero_fees=True)
detail0

,price_open,raw_signal_weight,market_return_from_prev_open,actual_weight_after_delay,weight_used_for_this_return,gross_return_contribution,portfolio_gross_returns,gross_equity,target_allocation_usd,target_shares,share_diff,total_costs_usd,net_equity
2024-01-01,100,1.000000,0.000000,1.000000,NaN,NaN,0.000000,100000.000000,100000.000000,1000.000000,0.000000,0.000000,100000.000000
2024-01-02,110,1.000000,0.100000,1.000000,1.000000,0.100000,0.100000,110000.000000,110000.000000,1000.000000,0.000000,0.000000,110000.000000
2024-01-03,99,-1.000000,-0.100000,-1.000000,1.000000,-0.100000,-0.100000,99000.000000,-99000.000000,-1000.000000,-2000.000000,0.000000,99000.000000
2024-01-04,120,0.000000,0.212121,0.000000,-1.000000,-0.212121,-0.212121,78000.000000,0.000000,0.000000,1000.000000,0.000000,78000.000000
2024-01-05,60,1.000000,-0.500000,1.000000,0.000000,-0.000000,0.000000,78000.000000,78000.000000,1300.000000,1300.000000,0.000000,78000.000000


Read the row `2024-01-03` carefully:

- price moved from `110` to `99`, so `market_return_from_prev_open = -10%`
- `weight_used_for_this_return` is yesterday's delayed weight
- the gross portfolio return is `weight_used_for_this_return * market_return_from_prev_open`

The engine always treats the row's return as already having happened between the previous row and the current row.

## Example 2: one asset, zero fees, `execution_delay=1`

With `execution_delay=1`, a signal stamped on `2024-01-01` becomes an actual target weight on `2024-01-02`, and earns the return shown on `2024-01-03`.

This matches the pattern: decide after close on day `t`, enter at open on day `t+1`, earn open-to-open return ending on day `t+2`.

In [4]:
engine1, stats1, detail1 = explain_engine(prices_1, signals_1, delay=1, zero_fees=True)
detail1

,price_open,raw_signal_weight,market_return_from_prev_open,actual_weight_after_delay,weight_used_for_this_return,gross_return_contribution,portfolio_gross_returns,gross_equity,target_allocation_usd,target_shares,share_diff,total_costs_usd,net_equity
2024-01-01,100,1.000000,0.000000,0.000000,NaN,NaN,0.000000,100000.000000,0.000000,0.000000,0.000000,0.000000,100000.000000
2024-01-02,110,1.000000,0.100000,1.000000,0.000000,0.000000,0.000000,100000.000000,100000.000000,909.090909,909.090909,0.000000,100000.000000
2024-01-03,99,-1.000000,-0.100000,1.000000,1.000000,-0.100000,-0.100000,90000.000000,90000.000000,909.090909,0.000000,0.000000,90000.000000
2024-01-04,120,0.000000,0.212121,-1.000000,1.000000,0.212121,0.212121,109090.909091,-109090.909091,-909.090909,-1818.181818,0.000000,109090.909091
2024-01-05,60,1.000000,-0.500000,0.000000,-1.000000,0.500000,0.500000,163636.363636,0.000000,0.000000,909.090909,0.000000,163636.363636


In [5]:
comparison = pd.DataFrame({
    "price": prices_1["ABC"],
    "signal": signals_1["ABC"],
    "gross_return_delay_0": stats0["gross_returns"],
    "gross_equity_delay_0": stats0["gross_equity"],
    "gross_return_delay_1": stats1["gross_returns"],
    "gross_equity_delay_1": stats1["gross_equity"],
})
comparison

,price,signal,gross_return_delay_0,gross_equity_delay_0,gross_return_delay_1,gross_equity_delay_1
2024-01-01,100,1.000000,0.000000,100000.000000,0.000000,100000.000000
2024-01-02,110,1.000000,0.100000,110000.000000,0.000000,100000.000000
2024-01-03,99,-1.000000,-0.100000,99000.000000,-0.100000,90000.000000
2024-01-04,120,0.000000,-0.212121,78000.000000,0.212121,109090.909091
2024-01-05,60,1.000000,0.000000,78000.000000,0.500000,163636.363636


## Example 3: alignment by date and ticker

`prices.align(signals, join="inner")` keeps only dates and columns present in both objects. It does not shift anything in time.

In [6]:
prices_misaligned = pd.DataFrame(
    {
        "ABC": [100, 101, 102, 103],
        "UNUSED_PRICE_ONLY": [50, 51, 52, 53],
    },
    index=pd.to_datetime(["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04"]),
)

signals_misaligned = pd.DataFrame(
    {
        "ABC": [1.0, -1.0, 0.0, 1.0],
        "UNUSED_SIGNAL_ONLY": [1.0, 1.0, 1.0, 1.0],
    },
    index=pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"]),
)

aligned_prices, aligned_signals = prices_misaligned.align(signals_misaligned, join="inner")

display(prices_misaligned)
display(signals_misaligned)
display(aligned_prices)
display(aligned_signals)

,ABC,UNUSED_PRICE_ONLY
2024-01-01,100,50
2024-01-02,101,51
2024-01-03,102,52
2024-01-04,103,53


,ABC,UNUSED_SIGNAL_ONLY
2024-01-02,1.000000,1.000000
2024-01-03,-1.000000,1.000000
2024-01-04,0.000000,1.000000
2024-01-05,1.000000,1.000000


,ABC
2024-01-02,101
2024-01-03,102
2024-01-04,103


,ABC
2024-01-02,1.000000
2024-01-03,-1.000000
2024-01-04,0.000000


## Example 4: multi-asset returns are summed across assets

For a multi-asset portfolio, each asset contributes:

```python
weight_used_for_return[asset] * market_return[asset]
```

The portfolio gross return for the row is the sum across columns.

In [7]:
prices_2 = pd.DataFrame(
    {
        "ABC": [100, 110, 99, 120, 60],
        "XYZ": [200, 190, 209, 209, 220],
    },
    index=dates,
)

signals_2 = pd.DataFrame(
    {
        "ABC": [0.50, 0.50, -0.50, 0.00, 0.50],
        "XYZ": [0.50, -0.25, -0.25, 0.25, 0.00],
    },
    index=dates,
)

engine2, stats2, detail2 = explain_engine(prices_2, signals_2, delay=1, zero_fees=True)

display(detail2["market_returns"])
display(detail2["weights_used_for_returns"])
display(detail2["gross_return_contributions"])
display(stats2[["gross_returns", "gross_equity", "net_equity"]])

,ABC,XYZ
2024-01-01,0.000000,0.000000
2024-01-02,0.100000,-0.050000
2024-01-03,-0.100000,0.100000
2024-01-04,0.212121,0.000000
2024-01-05,-0.500000,0.052632


,ABC,XYZ
2024-01-01,NaN,NaN
2024-01-02,0.000000,0.000000
2024-01-03,0.500000,0.500000
2024-01-04,0.500000,-0.250000
2024-01-05,-0.500000,-0.250000


,ABC,XYZ
2024-01-01,NaN,NaN
2024-01-02,0.000000,-0.000000
2024-01-03,-0.050000,0.050000
2024-01-04,0.106061,-0.000000
2024-01-05,0.250000,-0.013158


,gross_returns,gross_equity,net_equity
2024-01-01,0.000000,100000.000000,100000.000000
2024-01-02,0.000000,100000.000000,100000.000000
2024-01-03,0.000000,100000.000000,100000.000000
2024-01-04,0.106061,110606.060606,110606.060606
2024-01-05,0.236842,136802.232855,136802.232855


## Example 5: real transaction costs

The engine sizes target shares from the delayed target weights and gross equity:

```python
target_allocation_usd = actual_weights * gross_equity
target_shares = target_allocation_usd / price
share_diff = target_shares.diff().fillna(0)
```

Costs are charged on the absolute change in shares. This means changing from long to short is expensive because the share difference includes closing the long and opening the short.

In [8]:
engine_cost, stats_cost, detail_cost = explain_engine(
    prices_1,
    signals_1,
    delay=1,
    initial_capital=100_000,
    asset_class="STK",
    zero_fees=False,
)

detail_cost[[
    "price_open",
    "actual_weight_after_delay",
    "gross_equity",
    "target_allocation_usd",
    "target_shares",
    "share_diff",
    "total_costs_usd",
    "net_equity",
]]

,price_open,actual_weight_after_delay,gross_equity,target_allocation_usd,target_shares,share_diff,total_costs_usd,net_equity
2024-01-01,100,0.000000,100000.000000,0.000000,0.000000,0.000000,0.000000,100000.000000
2024-01-02,110,1.000000,100000.000000,100000.000000,909.090909,909.090909,13.181818,99986.818182
2024-01-03,99,1.000000,90000.000000,90000.000000,909.090909,0.000000,0.000000,89986.818182
2024-01-04,120,-1.000000,109090.909091,-109090.909091,-909.090909,-1818.181818,34.549091,109043.178182
2024-01-05,60,0.000000,163636.363636,0.000000,0.000000,909.090909,8.636364,163579.996364


## Example 6: mapping this to an after-close predictor

Suppose a signal stamped on date `t` is generated after the close of `t` using full day-`t` OHLCV data.

If the trade is intended to enter at `Open_{t+1}` and earn the return from `Open_{t+1}` to `Open_{t+2}`, then in this engine the signal must appear in the return row for `t+2`.

Because the engine computes row returns from previous open to current open, this intended behavior corresponds to `execution_delay=1`.

In [9]:
timing = pd.DataFrame({
    "open_price": prices_1["ABC"],
    "signal_generated_after_close": signals_1["ABC"],
    "return_row_is_prev_open_to_this_open": prices_1["ABC"].pct_change(fill_method=None).fillna(0),
    "delay_0_weight_used_for_return": detail0["weight_used_for_this_return"],
    "delay_1_weight_used_for_return": detail1["weight_used_for_this_return"],
})

timing

,open_price,signal_generated_after_close,return_row_is_prev_open_to_this_open,delay_0_weight_used_for_return,delay_1_weight_used_for_return
2024-01-01,100,1.000000,0.000000,NaN,NaN
2024-01-02,110,1.000000,0.100000,1.000000,0.000000
2024-01-03,99,-1.000000,-0.100000,1.000000,1.000000
2024-01-04,120,0.000000,0.212121,-1.000000,1.000000
2024-01-05,60,1.000000,-0.500000,0.000000,-1.000000


In the table above, the `2024-01-01` signal appears in:

- `delay_0_weight_used_for_return` on `2024-01-02`, meaning it earns `Open_2024-01-01 -> Open_2024-01-02`
- `delay_1_weight_used_for_return` on `2024-01-03`, meaning it earns `Open_2024-01-02 -> Open_2024-01-03`

For an after-close day-`t` predictor that trades at the next open, the second behavior is the one you want.